In [3]:
import torchaudio
print(torchaudio.list_audio_backends())
print(torchaudio.get_audio_backend())

import torchaudio
torchaudio.set_audio_backend("soundfile")

['soundfile']
None


C:\Users\smi09\AppData\Local\Temp\ipykernel_20084\136105733.py:3: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  print(torchaudio.get_audio_backend())
C:\Users\smi09\AppData\Local\Temp\ipykernel_20084\136105733.py:6: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [5]:
import torch
import torchaudio

# Load pretrained model
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()
model.eval()

# Load audio
waveform, sample_rate = torchaudio.load("Harvard.wav")

# Resample if needed
if sample_rate != bundle.sample_rate:
    waveform = torchaudio.functional.resample(
        waveform, sample_rate, bundle.sample_rate
    )

# Forward pass
with torch.no_grad():
    emissions, _ = model(waveform)

# Get labels
labels = bundle.get_labels()

# Greedy CTC decoding
emissions = torch.log_softmax(emissions, dim=-1)
pred_ids = torch.argmax(emissions, dim=-1)

# Collapse repeated tokens and remove blanks (CTC decoding)
transcript = ""
prev = -1
for idx in pred_ids[0]:
    idx = idx.item()
    if idx != prev and idx != 0:   # 0 is blank token
        transcript += labels[idx]
    prev = idx

print(transcript)

HARVARD|LIST|NUMBER|ONE|THE|BIRCH|CANOE|SLID|ON|THE|SMOOTH|PLANKS|GLEW|THE|SHEET|TO|THE|DARK|BLUE|BACKGROUND|IT'S|EASY|TO|TELL|THE|DEPTH|OF|A|WELL|THESE|DAYS|A|CHICKEN|LEG|IS|A|RARE|DISH|RICE|IS|OFTEN|SERVED|IN|ROUND|BOWLS|THE|JUICE|OF|LEMONS|MAKES|FINE|PUNCH|THE|BOX|WAS|THROWN|BESIDE|THE|PARKED|TRUCK|THE|HOX|WERE|FED|CHOPPED|CORN|AND|GARBAGE|FOUR|HOURS|OF|STEADY|WORK|FACED|OS|LARGE|SIZE|AND|STOCKINGS|IS|HARD|TO|SELL|
